# MLA (Multi-Head Latent Attention)

## 概述

MLA是DeepSeek-V2引入的创新注意力机制，通过低秩压缩KV缓存来显著降低推理时的内存占用。

### 核心问题

**KV缓存爆炸**:
```
标准多头注意力（MHA）:
KV缓存 = 2 × n_layers × n_heads × d_head × seq_len

例如 Llama-2-70B:
- 80层
- 64个头
- 每个头128维
- 序列长度4096

KV缓存 = 2 × 80 × 64 × 128 × 4096 × 4字节 = 25.6 GB!
```

### MLA的解决方案

**核心思想**: 所有头共享低秩的潜在表示

```
标准MHA: 每个头独立的KV
K, V = [n_heads × d_head] 维

MLA: 所有头共享压缩的KV
K_compressed, V_compressed = [d_latent] 维
K, V = expand(K_compressed, V_compressed)

压缩比: (n_heads × d_head) / d_latent
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. 标准多头注意力（对比基准）

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


class StandardMultiHeadAttention:
    """
    标准多头注意力
    
    每个头独立的KV投影
    KV缓存: n_heads × d_head × seq_len
    """
    
    def __init__(self, embed_dim, n_heads):
        assert embed_dim % n_heads == 0
        
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.d_head = embed_dim // n_heads
        
        # QKV投影
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def forward(self, x):
        seq_len, embed_dim = x.shape
        
        # 投影并reshape为多头
        Q = np.dot(x, self.W_q).reshape(seq_len, self.n_heads, self.d_head)
        K = np.dot(x, self.W_k).reshape(seq_len, self.n_heads, self.d_head)
        V = np.dot(x, self.W_v).reshape(seq_len, self.n_heads, self.d_head)
        
        # 转置为 (n_heads, seq_len, d_head)
        Q = Q.transpose(1, 0, 2)
        K = K.transpose(1, 0, 2)
        V = V.transpose(1, 0, 2)
        
        # 计算注意力
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_head)
        attention = softmax(scores, axis=-1)
        out = np.matmul(attention, V)
        
        # 合并多头
        out = out.transpose(1, 0, 2).reshape(seq_len, embed_dim)
        output = np.dot(out, self.W_o)
        
        return output, K, V


# 测试
embed_dim = 256
n_heads = 8
seq_len = 64

x = np.random.randn(seq_len, embed_dim)

std_mha = StandardMultiHeadAttention(embed_dim, n_heads)
output, K, V = std_mha.forward(x)

print(f"标准MHA:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  K缓存: {K.shape} (n_heads, seq_len, d_head)")
print(f"  V缓存: {V.shape}")
print(f"  KV缓存大小: {(K.size + V.size) * 4 / 1024:.2f} KB")

## 2. MLA实现

In [ ]:
class MultiHeadLatentAttention:
    """
    多头潜在注意力（MLA）
    
    核心创新:
    1. KV先压缩到低维潜在空间
    2. 所有头共享压缩后的KV
    3. 每个头独立扩展KV
    
    KV缓存: d_latent × seq_len
    """
    
    def __init__(self, embed_dim, n_heads, d_latent=None):
        assert embed_dim % n_heads == 0
        
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.d_head = embed_dim // n_heads
        self.d_latent = d_latent if d_latent else embed_dim // 4
        
        # Q投影（标准）
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        # KV压缩：embed_dim → d_latent
        self.W_k_compress = np.random.randn(embed_dim, self.d_latent) / np.sqrt(embed_dim)
        self.W_v_compress = np.random.randn(embed_dim, self.d_latent) / np.sqrt(embed_dim)
        
        # KV扩展：d_latent → embed_dim
        self.W_k_expand = np.random.randn(self.d_latent, embed_dim) / np.sqrt(self.d_latent)
        self.W_v_expand = np.random.randn(self.d_latent, embed_dim) / np.sqrt(self.d_latent)
        
        # 输出投影
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def forward(self, x):
        seq_len, embed_dim = x.shape
        
        # Q投影（标准）
        Q = np.dot(x, self.W_q).reshape(seq_len, self.n_heads, self.d_head)
        
        # KV压缩到潜在空间
        K_compressed = np.dot(x, self.W_k_compress)  # (seq_len, d_latent)
        V_compressed = np.dot(x, self.W_v_compress)  # (seq_len, d_latent)
        
        # KV扩展到每个头
        K = np.dot(K_compressed, self.W_k_expand).reshape(seq_len, self.n_heads, self.d_head)
        V = np.dot(V_compressed, self.W_v_expand).reshape(seq_len, self.n_heads, self.d_head)
        
        # 转置
        Q = Q.transpose(1, 0, 2)
        K = K.transpose(1, 0, 2)
        V = V.transpose(1, 0, 2)
        
        # 计算注意力
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_head)
        attention = softmax(scores, axis=-1)
        out = np.matmul(attention, V)
        
        # 合并多头
        out = out.transpose(1, 0, 2).reshape(seq_len, embed_dim)
        output = np.dot(out, self.W_o)
        
        return output, K_compressed, V_compressed


# 测试MLA
d_latent = embed_dim // 4  # 4x压缩
mla = MultiHeadLatentAttention(embed_dim, n_heads, d_latent=d_latent)
output_mla, K_comp, V_comp = mla.forward(x)

print(f"\nMLA (压缩率=4x):")
print(f"  输入: {x.shape}")
print(f"  输出: {output_mla.shape}")
print(f"  K_compressed缓存: {K_comp.shape} (seq_len, d_latent)")
print(f"  V_compressed缓存: {V_comp.shape}")
print(f"  KV缓存大小: {(K_comp.size + V_comp.size) * 4 / 1024:.2f} KB")
print(f"  缓存节省: {(1 - (K_comp.size + V_comp.size) / (K.size + V.size)) * 100:.1f}%")

## 3. KV缓存对比

In [ ]:
# 对比不同压缩率
compression_ratios = [2, 4, 8, 16]
results = []

# 标准MHA作为基准
_, K_std, V_std = std_mha.forward(x)
std_cache_size = (K_std.size + V_std.size) * 4 / 1024  # KB

print("\nKV缓存对比:")
print("-" * 70)
print(f"{'方法':<25} {'缓存大小(KB)':<20} {'节省比例':<15}")
print("-" * 70)
print(f"{'标准MHA':<25} {std_cache_size:<20.2f} {'基准':<15}")

for ratio in compression_ratios:
    d_latent = embed_dim // ratio
    mla = MultiHeadLatentAttention(embed_dim, n_heads, d_latent=d_latent)
    _, K_comp, V_comp = mla.forward(x)
    
    mla_cache_size = (K_comp.size + V_comp.size) * 4 / 1024
    saving = (1 - mla_cache_size / std_cache_size) * 100
    
    results.append({'ratio': ratio, 'size': mla_cache_size, 'saving': saving})
    print(f"{'MLA (压缩' + str(ratio) + 'x)':<25} {mla_cache_size:<20.2f} {saving:<15.1f}%")

print("-" * 70)

In [ ]:
# 可视化缓存大小
methods = ['标准MHA'] + [f'MLA-{r}x' for r in compression_ratios]
sizes = [std_cache_size] + [r['size'] for r in results]

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, sizes, color=['red'] + ['steelblue']*len(compression_ratios))

# 添加数值标签
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}',
             ha='center', va='bottom', fontsize=10)

plt.xlabel('方法', fontsize=12)
plt.ylabel('KV缓存大小 (KB)', fontsize=12)
plt.title(f'KV缓存对比 (序列长度={seq_len})', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4. 不同序列长度的缓存分析

In [ ]:
# 不同序列长度的缓存大小
seq_lengths = [128, 256, 512, 1024, 2048, 4096]

std_caches = []
mla_4x_caches = []
mla_8x_caches = []

for seq_len in seq_lengths:
    x_test = np.random.randn(seq_len, embed_dim)
    
    # 标准MHA
    _, K, V = std_mha.forward(x_test)
    std_size = (K.size + V.size) * 4 / (1024*1024)  # MB
    std_caches.append(std_size)
    
    # MLA 4x
    mla_4x = MultiHeadLatentAttention(embed_dim, n_heads, d_latent=embed_dim//4)
    _, K_comp, V_comp = mla_4x.forward(x_test)
    mla_4x_size = (K_comp.size + V_comp.size) * 4 / (1024*1024)
    mla_4x_caches.append(mla_4x_size)
    
    # MLA 8x
    mla_8x = MultiHeadLatentAttention(embed_dim, n_heads, d_latent=embed_dim//8)
    _, K_comp, V_comp = mla_8x.forward(x_test)
    mla_8x_size = (K_comp.size + V_comp.size) * 4 / (1024*1024)
    mla_8x_caches.append(mla_8x_size)

# 绘制
plt.figure(figsize=(12, 6))
plt.plot(seq_lengths, std_caches, 'o-', label='标准MHA', linewidth=2, markersize=8)
plt.plot(seq_lengths, mla_4x_caches, 's-', label='MLA-4x', linewidth=2, markersize=8)
plt.plot(seq_lengths, mla_8x_caches, '^-', label='MLA-8x', linewidth=2, markersize=8)

plt.xlabel('序列长度', fontsize=12)
plt.ylabel('KV缓存大小 (MB)', fontsize=12)
plt.title('KV缓存随序列长度的增长', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

# 打印数据
print("\n序列长度 vs KV缓存:")
print("-" * 70)
print(f"{'序列长度':<12} {'标准MHA(MB)':<15} {'MLA-4x(MB)':<15} {'节省比例':<12}")
print("-" * 70)

for i, seq_len in enumerate(seq_lengths):
    saving = (1 - mla_4x_caches[i] / std_caches[i]) * 100
    print(f"{seq_len:<12} {std_caches[i]:<15.4f} {mla_4x_caches[i]:<15.4f} {saving:<12.1f}%")

print("-" * 70)

## 5. 压缩可视化

In [ ]:
# 可视化KV压缩过程
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 标准MHA结构
ax1 = axes[0]
std_structure = np.random.rand(n_heads, embed_dim // n_heads)
im1 = ax1.imshow(std_structure, cmap='Blues', aspect='auto')
ax1.set_title('标准MHA的KV结构\n每个头独立的KV', fontsize=13, fontweight='bold')
ax1.set_xlabel('d_head', fontsize=11)
ax1.set_ylabel('n_heads', fontsize=11)
ax1.set_yticks(range(n_heads))
plt.colorbar(im1, ax=ax1)

# MLA结构
ax2 = axes[1]
mla_structure = np.random.rand(1, d_latent)
im2 = ax2.imshow(mla_structure, cmap='Greens', aspect='auto')
ax2.set_title(f'MLA的压缩KV\n所有头共享 (d_latent={d_latent})', fontsize=13, fontweight='bold')
ax2.set_xlabel('d_latent', fontsize=11)
ax2.set_ylabel('共享', fontsize=11)
ax2.set_yticks([0])
ax2.set_yticklabels(['All Heads'])
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("\n结构对比:")
print(f"标准MHA: {n_heads} 个头 × {embed_dim // n_heads} 维 = {embed_dim} 维")
print(f"MLA压缩: 1 个共享 × {d_latent} 维 = {d_latent} 维")
print(f"压缩比: {embed_dim / d_latent:.1f}x")

## 6. 输出质量分析

In [ ]:
# 比较不同压缩率的输出质量
x_test = np.random.randn(128, embed_dim)

# 标准MHA输出
output_std, _, _ = std_mha.forward(x_test)

print("输出质量对比:")
print("-" * 60)
print(f"{'压缩比':<12} {'平均误差':<15} {'最大误差':<15}")
print("-" * 60)

for ratio in [2, 4, 8, 16]:
    d_latent = embed_dim // ratio
    mla = MultiHeadLatentAttention(embed_dim, n_heads, d_latent=d_latent)
    output_mla, _, _ = mla.forward(x_test)
    
    diff = np.abs(output_std - output_mla)
    mean_err = np.mean(diff)
    max_err = np.max(diff)
    
    print(f"{ratio}x<12} {mean_err:<15.6f} {max_err:<15.6f}")

print("-" * 60)
print("\n注: MLA的权重是随机初始化的，实际使用时通过训练可以补偿压缩带来的误差")

## 7. 实际场景模拟：大模型推理

In [ ]:
# 模拟DeepSeek-V2的参数规模
print("\n大模型推理场景模拟 (DeepSeek-V2风格):")
print("=" * 70)

# 模型参数
n_layers = 60
embed_dim = 5120
n_heads = 128
batch_size = 1
max_seq_len = 4096

print(f"\n模型配置:")
print(f"  层数: {n_layers}")
print(f"  嵌入维度: {embed_dim}")
print(f"  头数: {n_heads}")
print(f"  批次大小: {batch_size}")
print(f"  最大序列长度: {max_seq_len}")

# 计算KV缓存大小
def calc_kv_cache(n_layers, n_heads, d_head, seq_len, batch_size):
    # 2 (K和V) × layers × batch × n_heads × seq_len × d_head × 4 (float32)
    return 2 * n_layers * batch_size * n_heads * seq_len * d_head * 4

def calc_mla_cache(n_layers, d_latent, seq_len, batch_size):
    # 2 (K和V) × layers × batch × seq_len × d_latent × 4 (float32)
    return 2 * n_layers * batch_size * seq_len * d_latent * 4

d_head = embed_dim // n_heads

print(f"\nKV缓存分析 (序列长度={max_seq_len}):")
print("-" * 70)

# 标准MHA
std_cache = calc_kv_cache(n_layers, n_heads, d_head, max_seq_len, batch_size)
print(f"标准MHA: {std_cache / (1024**3):.2f} GB")

# MLA不同压缩率
for ratio in [4, 8]:
    d_latent = embed_dim // ratio
    mla_cache = calc_mla_cache(n_layers, d_latent, max_seq_len, batch_size)
    saving = (1 - mla_cache / std_cache) * 100
    print(f"MLA-{ratio}x (d_latent={d_latent}): {mla_cache / (1024**3):.2f} GB (节省 {saving:.1f}%)")

print("-" * 70)

## 8. 总结

### MLA的核心优势

1. **显著降低KV缓存**: 5-10x减少
2. **加速推理**: 特别是长序列场景
3. **支持更长上下文**: 内存节省可用于更长序列
4. **易于部署**: 降低硬件要求

### 核心技术

- **低秩KV压缩**: 降维技术
- **共享潜在表示**: 所有头共享
- **两阶段设计**: 压缩 → 扩展
- **解耦位置编码**: RoPE单独处理

### 压缩比选择

| 压缩比 | 缓存节省 | 质量 | 适用场景 |
|--------|---------|------|----------|
| 2x | ~50% | 极高 | 要求最高质量 |
| 4x | ~75% | 高 | **推荐** |
| 8x | ~87.5% | 中 | 内存受限场景 |
| 16x | ~93.75% | 低 | 极端内存限制 |

### 实际应用

- **DeepSeek-V2**: 使用MLA处理长上下文
- **DeepSeek-V3**: 进一步优化
- **部署优化**: 降低推理成本

### 权衡

- ⚠️ 需要重新训练（不能直接应用于已有模型）
- ⚠️ 压缩率过高会影响性能
- ✅ 通过训练可以补偿大部分质量损失